Dec-POMDP Formulation & CTDE Architecture for MAPPO

In single-agent RL (PPO, SAC), the environment is assumed to be stationary: given state $s$ and action $a$, the probability distribution over the next state $s'$ is fixed.

Think of it like a video game

There are 2 agents:

        🌍 ENVIRONMENT

      A        B       TARGET
      🤖       🤖         🎯

The environment knows everything:

A's position
B's position
target position
walls
etc.

Call that state:

$$ s_t $$
What does Agent A receive?

Only the information we choose to show Agent A.

That's its observation:

$$ \boxed{o_A} $$

For example:

Environment knows:
A = (2,2)
B = (7,5)
Target = (9,9)

Agent A receives:
A's position = (2,2)
Target = (9,9)

So:

$$ o_A=[(2,2),(9,9)] $$

Maybe B isn't included.

What does Agent B receive?

B gets its own view:

Agent B receives:
B's position = (7,5)
Target = (9,9)

So:

$$ o_B=[(7,5),(9,9)] $$
Therefore
                 ENVIRONMENT
              knows everything
                     │
          ┌──────────┴──────────┐
          ↓                     ↓
       Agent A               Agent B
       gets o_A               gets o_B
          ↓                     ↓
       action A               action B

\(o_A\) literally means "what Agent A can see."

\(o_B\) literally means "what Agent B can see."

Nothing more complicated.

Why not just call them \(s_A\) and \(s_B\)?

Because they're not the environment's state.

There is one actual world:

$$ s_t $$

But different agents can receive different pieces/views of it:

$$ o_A = O_A(s_t) $$ $$ o_B = O_B(s_t) $$

For example:

$$ s_t = [\underbrace{A}_{2,2}, \underbrace{B}_{7,5}, \underbrace{Target}_{9,9}] $$

but:

$$ o_A=[A,Target] $$ $$ o_B=[B,Target] $$

That's literally the whole deal.

And if we give A information about B?

Then simply:

$$ o_A=[A,B,Target] $$

Now A can see B.

That's completely allowed. The observation definition is part of our environment design.

# 1. Start with the actual problem

Imagine two robots need to push a box through a door.

```text
        DOOR
         ↓
     ┌────────┐
     │  BOX   │
     └────────┘

   🤖 A       🤖 B
```

Both robots need to cooperate.

Now suppose A can only see:

```text
A's camera:
    BOX
     ↑
    🤖A
```

And B can only see its own surroundings.

Your question is:

> **If A doesn't know what B is doing, HOW THE HELL can A coordinate with B?**

Exactly.

## Answer: it doesn't necessarily need to know B's full state.

There are several ways coordination can happen.

---

# 2. Simplest case: the environment itself gives enough information

Suppose the rules are:

> **If A pushes left, B should push right.**

A doesn't need to know B's exact position.

It can learn:

```text
A sees:
box is on my right
        ↓
A → push right
```

B independently sees:

```text
box is on my left
        ↓
B → push left
```

They coordinate because their **local observations + learned policies + shared objective** produce compatible actions.

Think of two people carrying a table.

Person A doesn't necessarily need a live dashboard saying:

```text
B's exact hand position = (1.72, 3.41)
B's velocity = 0.83 m/s
B's intended action = ...
```

A can simply see enough of the table/person/environment to act appropriately.

---

# 3. But what if A REALLY needs to know B?

Now we have an important distinction.

There are two possibilities.

### Case A — A can observe B

Then B's position might simply be part of A's observation:

$$
o_A =
[\text{A position},\text{B position},\text{box position}]
$$

No problem.

A **does know about B**.

---

### Case B — A cannot observe B

Then:

$$
o_A =
[\text{A position},\text{box}]
$$

A genuinely doesn't know where B is.

Now coordination is harder.

But it can still learn coordination through **interaction history**.

For example:

```text
t=0
A does X
B does Y
→ reward +10

t=1
A does X
B does Z
→ reward -10
```

Over many episodes, A can learn:

> "When I see this situation, doing X tends to work."

It doesn't necessarily learn:

> "B is currently at coordinate (4,7)."

It learns a policy based on what **it can actually observe**.

---

# 4. Here's where your intuition is 100% correct

If B's hidden state is **essential** for A's decision, then A has a problem.

Imagine:

```text
A sees:

       BOX
        |
        A
```

But B could secretly be either:

```text
Situation 1:

B → BOX ← A


Situation 2:

B
↓
BOX ← A
```

A sees exactly the same thing.

But the correct action is different depending on B.

A cannot reliably choose the correct action.

Why?

Because **the information needed to make the decision isn't in \(o_A\)**.

No algorithm can magically recover information that isn't observable.

This is one of the fundamental problems of **partial observability**.

---

# 5. So how does MAPPO help?

Here's the clever part.

During **training**, we can give the critic information that the actors don't have.

Suppose the actual situation is:

```text
                 GLOBAL WORLD

        B
        ↓
       BOX ← A
```

The actors receive:

```text
A:
o_A = what A can see

B:
o_B = what B can see
```

But the critic receives:

```text
s = entire world
```

So:

```text
               GLOBAL STATE
                    │
                    ▼
             Central Critic
                  V(s)
                    │
                    │
        ┌───────────┴───────────┐
        ▼                       ▼
      Actor A                 Actor B
       o_A                     o_B
        ↓                       ↓
       a_A                     a_B
```

The critic knows:

> "Ah, B was actually behind the box."

A doesn't.

---

# 6. But wait — doesn't that STILL mean A can't coordinate?

**Yes, potentially.**

And this is extremely important:

> **CTDE does NOT magically solve partial observability.**

The centralized critic helps the **learning process**.

It does not give A hidden information at execution time.

If A genuinely needs B's hidden state to act correctly, you need something additional, such as:

* communication between agents
* better observations
* recurrent policies / memory
* belief states
* explicit communication actions

MAPPO itself isn't telepathy. 😭

---

# 7. Then why is the centralized critic useful?

Because training a partially observed multi-agent system is noisy as hell.

Consider A:

```text
A takes action LEFT
```

Then gets:

```text
reward = -10
```

Why?

Was A's action bad?

Or did B screw up?

Or was the environment already in a bad state?

A local critic only sees:

```text
o_A → V(o_A)
```

It doesn't have the whole picture.

Centralized critic sees:

```text
s
+
joint situation
→
V(s)
```

So it can produce a **better estimate of the team's situation**.

That gives A a better advantage estimate during training.

---

# 8. The key distinction

This is the thing I want you to lock in:

### Actor

Answers:

> **"Given what I can currently see, what should I do?"**

$$
a_i \sim \pi_i(a_i|o_i)
$$

### Critic

Answers:

> **"Given the whole situation, how good is this situation?"**

$$
V(s)
$$

So:

```text
EXECUTION:

A only gets → o_A → Actor A → a_A
B only gets → o_B → Actor B → a_B


TRAINING:

A's observation ─────→ Actor A
B's observation ─────→ Actor B
                           │
Global state ─────────→ Critic
                           │
                           ↓
                    better advantage
```

---

# 9. And now coordination makes more sense

Coordination doesn't necessarily mean:

> "A must know exactly what B is doing."

It can mean:

> "A and B have learned policies whose actions work well together."

For example, imagine traffic.

Two drivers don't need to know each other's neural-network internals.

They coordinate because they have:

* observations
* shared rules
* environment signals
* learned behavior
* possibly communication

Multi-agent RL is basically trying to learn this kind of **joint behavior**.

---

# 10. One final example

Imagine a game:

```text
A                B

🚗 →      🏁      ← 🚗
```

Reward:

```text
+100 if both reach the goal
-100 if they collide
```

A doesn't see B.

During training:

```text
A observation → Actor A → action A
B observation → Actor B → action B

                ↓
             Environment
                ↓
             reward
                ↓
       Global state → Critic
```

After millions of interactions, A may learn:

> "When I see this configuration, moving right tends to produce good team outcomes."

B independently learns its compatible behavior.

**That's coordination without direct knowledge of the other's internal state.**

But if the problem fundamentally requires hidden information from B, **we need communication or memory**. That's not a MAPPO failure; it's an information constraint.

---

if every agent gets the full global state, it's basically multi-agent PPO with centralized observations.

Classic MAPPO is specifically:

$$ \boxed{\text{Decentralized Actors} + \text{Centralized Critic}} $$

If actors also get the full state:

$$ \boxed{\text{Centralized Actors} + \text{Centralized Critic}} $$

So the PPO optimization machinery is still there, but you've removed the key decentralized-execution constraint that makes MAPPO interesting.

PettingZoo Multi-Agent Environment Setup

Yep. Now let's do the **actual 1.2 working**, one timestep at a time. No abstraction for the sake of abstraction.

# 1.2 — Actual Multi-Agent Environment Loop

We'll use a tiny 2-agent environment.

### Environment

```text
Position:
A = 2
B = 7
Target = 10
```

Each agent can choose:

```text
-1 = move left
 0 = stay
+1 = move right
```

Both agents are trying to reach the target.

---

## Step 0 — `reset()`

The environment starts:

$$
s_0=[2,7,10]
$$

Remember:

* `2` = A's position
* `7` = B's position
* `10` = target

We now generate each agent's observation.

Let's say we designed the environment so each agent sees **its own position + target**:

$$
o_A^0=[2,10]
$$

$$
o_B^0=[7,10]
$$

So the actors receive:

```text
A gets → [2, 10]
B gets → [7, 10]
```

They don't necessarily get the global state.

---

# Step 1 — Actors choose actions

Suppose the policies output:

$$
a_A^0=+1
$$

$$
a_B^0=+1
$$

Meaning both move right.

We send the **two actions together**:

```text
{
    A: +1,
    B: +1
}
```

This is the important difference from normal single-agent Gym.

---

# Step 2 — Environment executes them

The environment updates its state.

Our transition rule is:

$$
x_A^{t+1}=x_A^t+a_A^t
$$

$$
x_B^{t+1}=x_B^t+a_B^t
$$

Therefore:

$$
x_A^1=2+1=3
$$

$$
x_B^1=7+1=8
$$

Target remains 10.

So:

$$
\boxed{s_1=[3,8,10]}
$$

---

# Step 3 — Environment calculates reward

Let's make the reward:

$$
r_t=-\left(|x_A-target|+|x_B-target|\right)
$$

Before the action:

$$
|2-10|+|7-10|=8+3=11
$$

After the action:

$$
|3-10|+|8-10|=7+2=9
$$

So:

$$
\boxed{r_0=-9}
$$

Because we're using a shared team reward:

$$
r_A^0=r_B^0=-9
$$

The reward improved from \(-11\) to \(-9\), meaning the team moved closer.

---

# Step 4 — Generate new observations

The new state is:

$$
s_1=[3,8,10]
$$

Therefore:

$$
o_A^1=[3,10]
$$

$$
o_B^1=[8,10]
$$

Now we have completed **one RL timestep**.

Our collected transition is:

```text
t = 0

A observation = [2,10]
A action      = +1

B observation = [7,10]
B action      = +1

reward        = -9

next state    = [3,8,10]
```

---

# Step 5 — Do it again

At \(t=1\):

```text
A sees [3,10]
B sees [8,10]
```

Suppose:

$$
a_A=+1
$$

$$
a_B=+1
$$

Environment:

$$
x_A=3+1=4
$$

$$
x_B=8+1=9
$$

Therefore:

$$
s_2=[4,9,10]
$$

Reward:

$$
r_1=-(|4-10|+|9-10|)
$$

$$
=-7
$$

Again:

```text
A reward = -7
B reward = -7
```

And observations become:

$$
o_A^2=[4,10]
$$

$$
o_B^2=[9,10]
$$

---

# Step 6 — Eventually

Suppose:

```text
t=0 → reward -9
t=1 → reward -7
t=2 → reward -5
t=3 → reward -3
t=4 → reward ...
```

Eventually the agents reach the target.

The environment might then say:

```text
termination = True
```

and the episode ends.

---

# So what is PettingZoo doing?

**Basically this.**

We provide:

```text
actions:
{
    "agent_A": action_A,
    "agent_B": action_B
}
```

PettingZoo calls our environment's transition logic, and we return:

```text
observations
rewards
terminations
truncations
infos
```

The important mathematical loop is:

$$
\boxed{
s_t
\overset{a_A,a_B}{\longrightarrow}
s_{t+1}
}
$$

then:

$$
\boxed{
r_t=R(s_t,a_A,a_B,s_{t+1})
}
$$

then:

$$
\boxed{
s_{t+1}\rightarrow(o_A^{t+1},o_B^{t+1})
}
$$

And repeat.

---

## Where MAPPO eventually plugs in

For now:

```text
                ENVIRONMENT
                     │
              s_t → observations
                  ↙       ↘
               A           B
               ↓           ↓
              a_A         a_B
                  ↘       ↙
                 ENVIRONMENT
                     ↓
                  reward
```

Later, MAPPO adds:

```text
A observation → Actor A → a_A
B observation → Actor B → a_B

global state → Central Critic → V(s)
```
